In [ ]:
%run ./NB_Config_SELLING_SYSTEM

from datetime import datetime
import uuid
from delta.tables import DeltaTable
from pyspark.sql import functions as F

PIPELINE_RUN_ID = str(uuid.uuid4())
start_time = datetime.utcnow()
results = []

def write_gold(dataframe, table_name):
    gold_path = f'{GOLD_LH_ABFSS}/{GOLD_SCHEMA}/{table_name}'
    data_columns = [column_name for column_name in FINAL_COLUMNS[table_name]
                    if column_name in dataframe.columns]
    row_hash = F.sha2(
        F.concat_ws('||', *[
            F.coalesce(F.col(column_name).cast('string'), F.lit(''))
            for column_name in data_columns
        ]),
        256,
    )
    incoming = (dataframe
                .withColumn('_ROW_HASH', row_hash)
                .withColumn('_PIPELINE_NAME', F.lit(PIPELINE_NAME))
                .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
                .withColumn('_PUBLISHED_AT', F.current_timestamp()))
    incoming.createOrReplaceTempView(f'incoming_{table_name}')
    if not DeltaTable.isDeltaTable(spark, gold_path):
        incoming.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(gold_path)
    else:
        spark.sql(f"""
            MERGE INTO delta.`{gold_path}` AS target
            USING incoming_{table_name} AS source
            ON target._ROW_HASH = source._ROW_HASH
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.{table_name} "
        f"USING DELTA LOCATION '{gold_path}'"
    )
    return spark.read.format('delta').load(gold_path).count()

for table_name in FINAL_TABLES:
    silver_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{table_name}'
    dataframe = spark.read.format('delta').load(silver_path)
    row_count = write_gold(dataframe, table_name)
    results.append({'table': table_name, 'rows': row_count})
    print(f'{table_name}: merged {row_count:,} rows')

duration = (datetime.utcnow() - start_time).total_seconds()
print(f'Gold complete: {len(results)} tables | Duration: {duration:.1f}s')